# 04b – Model choice: the model race (README decisions 25–27)

**Question:** which model goes to the test set? In `04_models.ipynb` several variants lie within the noise of one split
(standard error about 3 points; with about 47 houses in the top 20% of a fold, one house is about 2 points). One split can
pick the winner by chance, and the more variants we compare, the more optimistic the best value is. So a small list of
candidates, fixed in the README **before** this run, competes on **five different splits** of the cross-validation data.

**Candidates** (from simple to complex, decision 25):
1. Ridge trained on houses only
2. Ridge trained on houses only, with compactness (1 / floors, 1 / √floor area)
3. Ridge with house interactions and compactness, trained on all buildings
4. Gradient boosting trained on all buildings (settings as in `04_models.ipynb`)

The two baselines (expert rule, training mean) run along as references, not as candidates.

**Five runs** (decision 26): run 1 uses the folds of `cv.csv` (the folds of `04_models.ipynb`); runs 2–5 split the same
12,710 buildings again into 5 folds with other seeds, with the same rules (groups together, same share of houses).
Every candidate is tuned again in every run (alpha or number of leaves, decision 14). The test set is not used.

**Rule** (decision 27): the leader has the highest mean selection measure over all 25 folds. Every other candidate is
compared with the leader fold by fold on the same houses. The runs reuse the same houses, so the 25 values are not
independent; we use the **corrected standard error** (Nadeau and Bengio): standard deviation × √(1/25 + 1/4).
If a simpler candidate is at most one corrected standard error behind the leader, the simplest such candidate wins.
The final model gets the setting with the highest mean over all 25 folds.

*Later this notebook becomes section 1 of `05_evaluation.ipynb`: first the choice, then the test set.*

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

from config import SEED, THRESHOLD, TABLES, FIGURES, load_cv
from pipeline import split_xy, ExpertRuleBaseline
from cv import (make_folds, run_variant, race_table, corrected_difference, choose_model, plot_race,
                best_alpha, SELECTION)

## 1. Candidates and runs

Settings as in `04_models.ipynb` (section 2): the same values of alpha and the same tree settings.

In [ ]:
RIDGE_ALPHAS = np.logspace(-2, 5, 29)   # 0.01 to 100,000, as in 04
LEAF_VALUES = [7, 15, 31]               # tree size, as in 04
TREE_KWARGS = {"learning_rate": 0.05, "max_iter": 300, "early_stopping": False, "random_state": SEED}

CANDIDATES = {   # from simple to complex (decision 27)
    "Ridge, houses only": {"model": Ridge, "tune": ("alpha", RIDGE_ALPHAS), "train_on": "houses"},
    "Ridge, houses only, compactness": {"model": Ridge, "tune": ("alpha", RIDGE_ALPHAS), "train_on": "houses",
                                        "switches": {"compactness": True}},
    "Ridge, house interactions, compactness": {"model": Ridge, "tune": ("alpha", RIDGE_ALPHAS),
                                               "switches": {"house_interactions": True, "compactness": True}},
    "Gradient boosting, reference": {"model": HistGradientBoostingRegressor, "tune": ("max_leaf_nodes", LEAF_VALUES),
                                     "kwargs": TREE_KWARGS},
}
REFERENCES = {   # baselines, not candidates
    "Expert rule, baseline": {"model": ExpertRuleBaseline, "baseline": True},
    "Training mean, baseline": {"model": DummyRegressor, "baseline": True},
}
SEEDS = {1: None, 2: SEED + 1, 3: SEED + 2, 4: SEED + 3, 5: SEED + 4}   # run 1: the folds of cv.csv

train = load_cv()
X, y, is_house, groups = split_xy(train)
print("Buildings:", len(train), "| houses:", is_house.sum())

## 2. Five splits of the cross-validation data

Run 1 is the split everyone has used so far. For runs 2–5 the same buildings are split again. The table checks every
split: houses per fold, share of houses above 450 per fold (not balanced on purpose, as in decision 10), and groups
split across folds (must be 0).

In [ ]:
folds_by_run = {run: train["fold"].to_numpy() if seed is None else make_folds(train, seed)
                for run, seed in SEEDS.items()}
print("make_folds with the project seed gives the folds of cv.csv:",
      bool((make_folds(train, SEED) == folds_by_run[1]).all()))

rows = []
for run, folds in folds_by_run.items():
    split = train.assign(fold=folds)
    houses = split[split["house"]]
    rows.append({
        "run": run, "seed": SEED if SEEDS[run] is None else SEEDS[run],
        "houses per fold": "–".join(str(n) for n in houses.groupby("fold").size()),
        "houses above 450 per fold (%)": "–".join(f"{v:.0f}" for v in houses.groupby("fold")["y"].apply(lambda s: (s > THRESHOLD).mean() * 100)),
        "groups split across folds": int((split.groupby("group_id")["fold"].nunique() > 1).sum()),
        "same fold as in run 1 (%)": round((folds == folds_by_run[1]).mean() * 100, 1),
    })
pd.DataFrame(rows).set_index("run")

**Reading the table.** By chance about 20% of the buildings land in the same fold number as in run 1; the folds are
really new. Every run keeps the groups together.

## 3. The race

Every candidate and baseline through every run: tune, then cross-validate with the chosen setting. Takes a few minutes.

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
results = {name: {} for name in {**CANDIDATES, **REFERENCES}}
for run, folds in folds_by_run.items():
    for name, spec in {**CANDIDATES, **REFERENCES}.items():
        r = run_variant(spec, X, y, is_house, folds)
        r["per_fold"].insert(0, "run", run)
        if r["tuning"] is not None:
            r["tuning"].insert(0, "run", run)
        results[name][run] = r
    print(f"run {run}:", " | ".join(f"{name}: {results[name][run]['value']:.3g}" for name in CANDIDATES
                                    if not np.isnan(results[name][run]["value"])))

per_fold = {name: pd.concat([results[name][run]["per_fold"] for run in SEEDS], ignore_index=True) for name in results}

**Check:** run 1 uses the same folds and settings as `04_models.ipynb`, so it must give the same numbers.

In [ ]:
summary_04 = pd.read_csv(TABLES / "models_cv_summary.csv").query("group == 'houses'").set_index("variant")
for name in results:
    run1 = per_fold[name].query("run == 1 and group == 'houses'")[SELECTION].mean()
    print(f"{name:<42} run 1: {run1 * 100:5.1f}% | 04_models: {summary_04.loc[name, SELECTION + '_mean'] * 100:5.1f}%")

## 4. Results

Mean over all 25 folds with its corrected standard error, the mean of every run, and the other measures on houses.

In [ ]:
race = race_table(per_fold)
race

**Fold by fold against the leader.** Positive = better than the leader. `better on average in` counts the runs in which
the candidate is ahead on average.

In [ ]:
order = list(CANDIDATES)
means = race.loc[order, "top 20%: above 450 (%)"]
leader = means.idxmax()
differences = pd.DataFrame([{"variant": name, "compared with": leader,
                             **corrected_difference(per_fold[name], per_fold[leader])[1]}
                            for name in results if name != leader]).set_index("variant")
differences

## 5. Figure

Small dots: mean of one run. Large dot: mean over all runs ± corrected standard error. Grey lines: references.

In [ ]:
random_share = (y[is_house] > THRESHOLD).mean() * 100
reference = {f"random choice ({random_share:.1f}%)": random_share,
             f"expert rule ({race.loc['Expert rule, baseline', 'top 20%: above 450 (%)']:.1f}%)":
                 race.loc["Expert rule, baseline", "top 20%: above 450 (%)"]}
plot_race({name: per_fold[name] for name in CANDIDATES}, reference=reference, path=FIGURES / "choice_race.png")
plt.show()

## 6. The decision (rule of decision 27)

In [ ]:
winner, leader, decision = choose_model({name: per_fold[name] for name in CANDIDATES}, order)
display(decision)
print("Leader (highest mean):", leader)
print("Winner (simplest candidate within one corrected SE of the leader):", winner)

## 7. Setting of the final model

The tuning curves of the five runs are averaged; the value with the highest mean over all 25 folds wins (decision 14,
MAE on houses as tie-break).

In [ ]:
param = CANDIDATES[winner]["tune"][0]
tuning = pd.concat([results[winner][run]["tuning"] for run in SEEDS], ignore_index=True)
measures = [c for c in tuning.columns if c.endswith("_mean")]
tuning_mean = tuning.groupby([param, "group"], as_index=False)[measures].mean()
final_value = best_alpha(tuning_mean, param=param)

per_run = {run: results[winner][run]["value"] for run in SEEDS}
print("Chosen in each run:", {run: round(v, 3) for run, v in per_run.items()})
print(f"Final setting: {param} = {final_value:.4g}")
curve = tuning_mean.query("group == 'houses'")[[param, SELECTION + "_mean", "mae_mean"]].copy()
curve[SELECTION + "_mean"] = (curve[SELECTION + "_mean"] * 100).round(1)
curve.round(2).set_index(param).T

## 8. Save

- `output/tables/choice_per_fold.csv`: every candidate and baseline, every run and fold
- `output/tables/choice_summary.csv`: the results table of section 4
- `output/tables/choice_differences.csv`: every variant against the leader (section 4)
- `output/tables/choice_decision.csv`: the decision table of section 6
- `output/tables/choice_final_model.csv`: the winner and its setting, input for the test set (task 26)
- `output/figures/choice_race.png`

In [ ]:
pd.concat(per_fold, names=["variant"]).reset_index(level=0).to_csv(TABLES / "choice_per_fold.csv", index=False)
race.to_csv(TABLES / "choice_summary.csv")
differences.to_csv(TABLES / "choice_differences.csv")
decision.to_csv(TABLES / "choice_decision.csv")
spec = CANDIDATES[winner]
pd.DataFrame([{"variant": winner, "model": spec["model"].__name__, "trained on": spec.get("train_on", "all"),
               "switches": json.dumps(spec.get("switches", {})), "fixed arguments": json.dumps(spec.get("kwargs", {})),
               "setting": param, "value": final_value}]).to_csv(TABLES / "choice_final_model.csv", index=False)
print("Saved the choice tables; winner:", winner, f"({param} = {final_value:.4g})")

## 9. What we found

- **All four candidates lie within one corrected standard error of each other** (58.1% to 60.1%, corrected SE 4–5 points);
  all beat the expert rule (44.7%) and random choice (38%) clearly, in every run.
- **Leader: ridge on houses only with compactness, 60.1%.** It is also the most stable: 58.7% to 61.3% over the five runs.
  Ridge on houses only without compactness is 1.6 ± 1.7 points behind; it is worse in 14 and better in 4 of 25 folds,
  and ahead on average in only 1 of 5 runs. So compactness is consistently a little better, but the lead is within one
  corrected standard error.
- **Winner by the rule: ridge trained on houses only** (the simplest candidate within one corrected SE of the leader),
  with alpha = 31.6 (best mean over all 25 folds; the chosen alpha moves between 0.06 and 31.6 from run to run, because the
  curve is flat).
- **One split can mislead.** Gradient boosting had the best value on the split of `04_models.ipynb` (61.7%); on the other
  four splits it reaches 55.7% to 62.1%, 58.8% on average. Ridge on houses only also had its best value on that split
  (60.4% vs. 58.5% on average). The race removes this luck of one split from the choice.
- **Open for the coaching:** the rule picks the simpler model although compactness is ahead in 4 of 5 runs. The final
  choice is made after the coaching and replaces README decision 19.